In [3]:
# All Necessary Imports
import numpy as np
import pandas as pd
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import re
from textblob import TextBlob
from wordcloud import WordCloud
import seaborn as sns
import matplotlib.pyplot as plt
import cufflinks as cf
%matplotlib inline
from plotly.offline import init_notebook_mode, iplot
init_notebook_mode(connected = True)
cf.go_offline();
import plotly.graph_objs as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns",None)
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Download NLTK data (run this only once)
nltk.download('stopwords')
nltk.download('wordnet')

# Load and Prepare the Data
df = pd.read_csv("amazon.csv")
df = df.sort_values("wilson_lower_bound", ascending=False)
df.drop("Unnamed: 0", inplace=True, axis=1)

# Advanced Data Cleaning and Column Creation
def advanced_clean_text(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = text.lower()
    words = text.split()
    stop_words = set(stopwords.words('english'))
    lemmatizer = WordNetLemmatizer()
    cleaned_words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]
    return " ".join(cleaned_words)

df['reviewText_cleaned'] = df['reviewText'].apply(advanced_clean_text)

# Regression Model Training
# Select features (X) and target (y)
X_reg = df['reviewText_cleaned']
y_reg = df['overall']

# Split data into training and testing sets
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

# Vectorize the text data
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_vec_reg = vectorizer.fit_transform(X_train_reg)
X_test_vec_reg = vectorizer.transform(X_test_reg)

# Initialize and train the Decision Tree Regressor model
regressor_model = DecisionTreeRegressor(random_state=42)
regressor_model.fit(X_train_vec_reg, y_train_reg)

print("Decision Tree Regressor model trained successfully!")

# Model Evaluation
# Make predictions on the test data
y_pred_reg = regressor_model.predict(X_test_vec_reg)

# Evaluate the model using Mean Absolute Error (MAE) and Root Mean Squared Error (RMSE)
mae = mean_absolute_error(y_test_reg, y_pred_reg)
mse = mean_squared_error(y_test_reg, y_pred_reg)
rmse = np.sqrt(mse)

print(f"\nMean Absolute Error (MAE): {mae:.2f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\sathv\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\sathv\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


Decision Tree Regressor model trained successfully!

Mean Absolute Error (MAE): 0.55
Root Mean Squared Error (RMSE): 1.18
